# Rag Pipelines - Data Ingestion to Vector DB Pipeline

### Data Process or loading

In [19]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [20]:

### Read all the pdf insede the directory.

def process_all_pdf(pdfs_directory):
    pdf_dir = Path(pdfs_directory)
    # print(pdf_dir)
    all_documents = []

    ### find all pdf files and list it.
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    # for pdfs in pdf_files:
    #     print(pdfs)

    print(f"Found {len(pdf_files)} pdf files here")

    ### processing every pdf files in this loop
    for pdf_file in pdf_files:
        print(f"\n Processing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()

            ### Add some metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages successfully.")
        except Exception as e:
            print(f"Error : {e}")
    print(f"Total Documents Loaded {len(all_documents)}")
    return all_documents
        


all_pdf_reader_Documents = process_all_pdf("../data")
# all_pdf_reader_Documents

Found 7 pdf files here

 Processing : Examining-Artificial.pdf
Loaded 17 pages successfully.

 Processing : file-example_PDF_1MB.pdf
Loaded 30 pages successfully.

 Processing : file-example_PDF_500_kB.pdf
Loaded 5 pages successfully.

 Processing : file-sample_150kB.pdf
Loaded 4 pages successfully.

 Processing : GED 201  3.pdf
Loaded 9 pages successfully.

 Processing : UniID card.pdf
Loaded 2 pages successfully.

 Processing : 📄 Full Stack Project Requirements.pdf
Loaded 3 pages successfully.
Total Documents Loaded 70


### Text spilitting get into chunks 

In [21]:
def split_all_document(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better Rag performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks=split_all_document(all_pdf_reader_Documents)
# print()
# print(len(chunks))
# for chunk in chunks:
#     print(chunk.metadata)


Split 70 documents into 179 chunks

Example chunk:
Content: ISSN: 2182-2069 / E-ISSN: 2182-2077 
99 
Examining Artificial Intelligence and Law as a Tool for Legal 
Service, Decision-making, Job Transformation, and Ethical 
Performance 
 
Carlos Ignacio Gallo A...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-09-10T15:01:26+05:30', 'author': 'JISIS', 'moddate': '2024-09-10T15:01:26+05:30', 'source': '..\\data\\pdf\\Examining-Artificial.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'Examining-Artificial.pdf', 'file_type': 'pdf'}


### Embedding And VectorStoreDB

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer          ### our embedding models will be available inside this. It is a library to download models from hugging face
import chromadb
from chromadb.config import Settings
import uuid                                                    ### it will generate an id for every record in the vectorDB
from typing import List,Dict,Tuple,Any
from sklearn.metrics.pairwise import cosine_distances

In [23]:
class Embedding_Manager:
    """Handles document embidding generation using SentenceTransformer"""
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loding embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)           ### Downloads the model from Hugging Face (first time) and Loads it into memory.
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self,texts:list[str]) -> np.ndarray:
        """
        generate embeddings for a list of texts
        return numpy array of embeddings with shape (len(texts), embeddings_dim) 
        
        """
        if not self.model:
            raise ValueError("Model is not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = Embedding_Manager()
embedding_manager

Loding embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7012.86it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [24]:
class VectorStore:
    """ Manages document embeddings in a ChromaDB vector store """
    def __init__(self, collection_name: str="pdf_documents", persist_directory: str= "../data/vector_store"):
        """ 
        Initialize the vectore store
            collection_name = name of the ChromaDB collection
            persist_directory = created vector store will be save in the hard disk of the device
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """ Initialize chromaDb client and collection """
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document emnbeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: list[Any], embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Meanings:
                documnets : list of LangChain documents
                embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Addings {len(documents)} documents to vector store.....")

        ## prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ## generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            ## prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            ## Document contenct
            documents_text.append(doc.page_content)

            ## Embeddings
            embeddings_list.append(embedding.tolist())
        
        ## Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas= metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 358


In [25]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-09-10T15:01:26+05:30', 'author': 'JISIS', 'moddate': '2024-09-10T15:01:26+05:30', 'source': '..\\data\\pdf\\Examining-Artificial.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'Examining-Artificial.pdf', 'file_type': 'pdf'}, page_content='ISSN: 2182-2069 / E-ISSN: 2182-2077 \n99 \nExamining Artificial Intelligence and Law as a Tool for Legal \nService, Decision-making, Job Transformation, and Ethical \nPerformance \n \nCarlos Ignacio Gallo Aguila1, María Del Pilar Castro Arellano2,                                           \nMaría Del Pilar Quezada Castro3, Eliana Maritza Barturen Mondragón4, and                                                                                    \nGuillermo Alexander Quezada Castro5* \n \n1Professor, Universidad César Vallejo, Perú. cgalloa@gmail.com,                            \nhttps://orcid.org/0000-0003-1382-0545 \n

In [26]:
## Convert the text into Embeddings.

texts = [doc.page_content for doc in chunks]
# texts

## Generate the Embeddings.
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vector_store.add_documents(chunks,embeddings)

Generating embeddings for 179 texts...


Batches: 100%|██████████| 6/6 [00:03<00:00,  1.63it/s]


Generated embeddings with shape: (179, 384)
Addings 179 documents to vector store.....
Successfully added 179 documents to vector store
Total documents in collection : 537


$$$

# Retriever Pipeline From VectorStore

In [27]:
class RAGRetriever:
    """ Handles query-based retrieval from the vector store """

    def __init__(self, vector_store:VectorStore, embedding_manager: Embedding_Manager):
        """
        Initialize the retriever
        Meaning: 
                vector_store: vector store containing documnet embeddings
                embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query:str, top_key: int=5, score_threshold: float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query

        Meanings:
                query : The search query
                top_key : Number of topresults to return
                score_threshold : Minimum similarity score threshold

        Returns:
                List of dictionaries containing retrieved documnets and metadata

          """
        print(f"Retrieving documents for query: '{query}' ")
        print(f"Top K: {top_key}, Score Threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_key
            )

            # process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    ## convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                                'id' : doc_id,
                                'content' : document,
                                'metadata' : metadata,
                                'similarity_score' : similarity_score,
                                'distance' : distance,
                                'rank' : i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documnets found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store,embedding_manager)

In [28]:
rag_retriever

In [29]:
rag_retriever.retrieve("What is Assets")

Retrieving documents for query: 'What is Assets' 
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_aeaafd8b_167',
  'content': 'and have the physical existence. \n2. Right for receiving payment:  Accounts Receivable, Sundry Debtors, Notes \nReceivable, advances to employees. \n3. Right for receiving services: Prepaid insurances, unexpired rates etc. \n4. Intangible assets: Goodwill, patents etc that has no physical existence. \n \nIn another sense assets can be classified into two groups: \n1. Long-term assets: Those assets that will give benefit more than one-year .For \nexample: Land, Equipment etc. \n \n2. Short term assets /Current assets: Those assets that will be matured within \none year. For example: Cash, bank balance, Accounts receivable etc. \n   \n \nLiabilities:  \n-Claims in the assets by outsiders or creditors  \n -Obligation to pay for purchases  \n \nLiabilities can be classified into two groups:',
  'metadata': {'creator': 'Microsoft® Word LTSC',
   'total_pages': 9,
   'content_length': 780,
   'page_label': '4',
   'author': 'MD Asraful Hasan',
   'f

## Integration Vectordb Context pipeline With LLM output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (Set your GORQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def simple_RAG(query, retriever, llm, top_key=3):
    ## retrievee the context
    result = retriever.retrieve(query,top_key=top_key)
    context="\n\n".join([doc['content'] for doc in result]) if result else ""
    if not context:
        return "No relevant context found to answer the query."
    ## generate the response using the groq LLM
    prompt=f""" use the following context to answer the query. If the context does not contain enough information, 
    respond with 'Insufficient information to answer the query.'
    Context: {context}
    Query:  {query}
    Answer: """
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [33]:
answer = simple_RAG("What is Assets", rag_retriever, llm)
print(f"Answer: {answer}")

Retrieving documents for query: 'What is Assets' 
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Assets are items or resources that have economic value and are owned or controlled by an individual or organization. They can be classified into two main groups:

1. Tangible assets: These are physical items that have a physical existence, such as:
   - Land
   - Equipment
   - Vehicles
   - Buildings

2. Intangible assets: These are non-physical items that have value but no physical existence, such as:
   - Goodwill
   - Patents
   - Copyrights
   - Trademarks

Assets can also be classified into two groups based on their duration:
1. Long-term assets: These are assets that will provide benefits for more than one year, such as:
   - Land
   - Equipment
   - Buildings

2. Short-term assets / Current assets: These are assets that will be used up or converted into cash within one year, such as:
   - Cash
   - Bank balance
   - Accounts receivable
   - Inventory


# Enhanced RAG Pipeline Features

In [39]:
# Enhanced RAG pipeline 
def rag_advanced(query, retriever, llm, top_key=5, min_score=0.2, return_context=False):
    """
     RAG pipeline with extra features:
        Return answer, sources, confidence score, optionally full context.
       """
    results = retriever.retrieve(query, top_key=top_key, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant context found to answer the query.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }
    # prepare context for LLM
    context="\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300] + "..."
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    # generate the response using the groq LLM
    prompt=f""" use the following context to answer the question concisely. \n\n{context} \n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }
    if return_context:
        output["context"] = context
    return output

# Example usage of the advanced RAG pipeline
result = rag_advanced("What is the Balance Sheet?", rag_retriever, llm, top_key=3, min_score=0.1, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence Score: {result['confidence']}")
print(f"Context: {result['context'][:300]}...")  # Print first 300 characters of context

    

Retrieving documents for query: 'What is the Balance Sheet?' 
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.31it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The Balance Sheet is a financial statement that presents the total assets of a firm and the total equities (liabilities and owner's equity) at a specific point in time, reflecting the fundamental accounting equation: A = E.
Sources: [{'source': 'GED 201  3.pdf', 'page': 2, 'score': 0.11171025037765503, 'preview': '● Trial balance—the list of all the ledger accounts with their balances \n \n \nDiagram: Accounting Process \n \nDouble Entry System and Accounting Equation: \n \nDouble entry bookkeeping is a generally accepted system. This system was first described \nby Luca Pacioli in 1494. The system of double entry ma...'}, {'source': 'GED 201  3.pdf', 'page': 2, 'score': 0.11171025037765503, 'preview': '● Trial balance—the list of all the ledger accounts with their balances \n \n \nDiagram: Accounting Process \n \nDouble Entry System and Accounting Equation: \n \nDouble entry bookkeeping is a generally accepted system. This system was first described \nby Luca Pacioli in 1494. 